In [68]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

print(" Loading consolidated raw university database...")
df = pd.read_csv("university_raw_data.csv", low_memory=False)
print(f"Loaded Raw Shape: {df.shape}")

 Loading consolidated raw university database...
Loaded Raw Shape: (675, 101)


In [69]:
# 1. Collapse duplicate join suffixes safely
df['country'] = df['Country_x'].fillna(df.get('Country Code', np.nan)).fillna('Unknown')

for base_col in ['city', 'region', 'type']:
    x_col = f'{base_col}_x'
    y_col = f'{base_col}_y'
    
    s_base = df[base_col] if base_col in df.columns else pd.Series(np.nan, index=df.index)
    s_x = df[x_col] if x_col in df.columns else pd.Series(np.nan, index=df.index)
    s_y = df[y_col] if y_col in df.columns else pd.Series(np.nan, index=df.index)
    
    if base_col == 'type':
        df['university_type'] = s_base.fillna(s_x).fillna(s_y).fillna('Unknown')
    else:
        df[base_col] = s_base.fillna(s_x).fillna(s_y).fillna('Unknown')

# Handle faculty count duplicate fields safely
s_fac = df['faculty_count'] if 'faculty_count' in df.columns else pd.Series(np.nan, index=df.index)
s_fac_x = df['faculty_count_x'] if 'faculty_count_x' in df.columns else pd.Series(np.nan, index=df.index)
s_fac_y = df['faculty_count_y'] if 'faculty_count_y' in df.columns else pd.Series(np.nan, index=df.index)
df['faculty_count'] = s_fac.fillna(s_fac_x).fillna(s_fac_y).fillna(0)

In [70]:
# 2. Clean Missing Value Tokens across continuous metrics
sentinels = [-1, -1.0, '-1', '-1.0', 'Unknown', 'unknown', 'NaN', 'nan']

numeric_metrics = [
    'Academic Reputation Score', 'Employer Reputation Score', 'Faculty Student Score', 
    'Citations per Faculty Score', 'International Faculty Score', 'International Students Score', 
    'International Research Network Score', 'Employment Outcomes Score', 'Sustainability Score',
    'Overall SCORE', 'Overall Score', 'Teaching', 'Research Environment', 'Research Quality', 
    'Industry Impact', 'International Outlook', 'citations_score', 'research_output_score',
    'country_avg_overall_score', 'country_avg_academic_reputation', 'country_avg_citations', 
    'country_avg_international_ratio', 'research_productivity_index', 'female_percentage', 'male_percentage'
]

for col in numeric_metrics:
    if col in df.columns:
        df[col] = df[col].replace(sentinels, np.nan)
        df[col] = pd.to_numeric(df[col], errors='coerce')

In [71]:
# 3. Clean Integer Value Elements
integer_metrics = [
    'Student Population', 'Students to Staff Ratio', 'Citations_Count', 
    'universities_ranked_count', 'best_university_rank'
]
for col in integer_metrics:
    if col in df.columns:
        if df[col].dtype == 'object':
            df[col] = df[col].astype(str).str.replace(',', '', regex=False)
        df[col] = df[col].replace(sentinels, np.nan)
        df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0)

# 4. Standardize ranks from text bands to whole numbers
for rank_col in ['World Rank', 'National_Rank_x']:
    if rank_col in df.columns:
        df[rank_col] = df[rank_col].astype(str).str.replace('+', '', regex=False)
        df[rank_col] = df[rank_col].str.split('-').str[0]
        df[rank_col] = pd.to_numeric(df[rank_col], errors='coerce').fillna(999)

In [72]:
# 5. Parse Gender Ratios into explicit features if empty
if 'female_percentage' not in df.columns:
    df['female_percentage'] = np.nan
if 'male_percentage' not in df.columns:
    df['male_percentage'] = np.nan

if 'Female to Male Ratio' in df.columns and pd.Series(df['female_percentage']).isnull().all():
    def extract_female(val):
        if pd.isna(val) or not isinstance(val, str): return 50.0
        parts = [p.strip() for p in val.split(':') if p.strip().isdigit()]
        if len(parts) >= 2:
            tot = float(parts[0]) + float(parts[1])
            return round((float(parts[0]) / tot) * 100, 2) if tot > 0 else 50.0
        return 50.0
    df['female_percentage'] = df['Female to Male Ratio'].apply(extract_female)
    df['male_percentage'] = 100 - df['female_percentage']

# Apply clean contextual standard fallbacks
df['university_name'] = df['University'].fillna('Unknown') if 'University' in df.columns else 'Unknown'
df['subject_field'] = df['subject_field'].fillna('General') if 'subject_field' in df.columns else 'General'
df['gender_ratio'] = df['Female to Male Ratio'].fillna('50:50') if 'Female to Male Ratio' in df.columns else '50:50'
df['degree_level'] = df['degree_level'].fillna('Undergraduate & Postgraduate') if 'degree_level' in df.columns else 'Undergraduate & Postgraduate'

In [73]:
# 6. Establish 38-column target schema structure blueprint
target_schema_38 = [
    'university_id', 'university_name', 'Year', 'World Rank', 'National_Rank_x', 
    'Overall SCORE', 'country', 'region', 'city', 'university_type',
    'Academic Reputation Score', 'Employer Reputation Score', 'citations_score',
    'Research_Output_x', 'Citations_Count', 'Citations per Faculty Score', 
    'research_output_score', 'research_productivity_index', 'subject_field', 
    'Student Population', 'International Students', 'faculty_count', 
    'Students to Staff Ratio', 'gender_ratio', 'female_percentage', 'male_percentage', 
    'degree_level', 'Enrollment', 'country_avg_rank', 'universities_ranked_count', 
    'best_university_rank', 'country_avg_overall_score', 'country_avg_academic_reputation', 
    'country_avg_citations', 'country_avg_international_ratio', 'Teaching', 
    'Research Environment', 'Research Quality'
]

# Guarantee missing placeholder space columns exist safely
for item in target_schema_38:
    if item not in df.columns:
        df[item] = np.nan

df_output = df[target_schema_38].copy()

# Rename columns to final system headers aligned with requirements
df_output.rename(columns={
    'Year': 'year',
    'World Rank': 'world_rank',
    'National_Rank_x': 'national_rank',
    'Overall SCORE': 'overall_score',
    'Research_Output_x': 'publications_count',
    'Student Population': 'total_students',
    'Students to Staff Ratio': 'faculty_to_student_ratio',
    'International Students': 'international_students_count'
}, inplace=True)

df_output.drop_duplicates(subset=['university_id', 'year'], inplace=True)
print(f"Sanity Check Columns Count: {df_output.shape[1]} columns ready.")

Sanity Check Columns Count: 38 columns ready.


In [74]:
# Show structural info to verify data types are clean for metric generation
df_output.info()

# Preview top rows
df_output.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 675 entries, 0 to 674
Data columns (total 38 columns):
 #   Column                           Non-Null Count  Dtype  
---  ------                           --------------  -----  
 0   university_id                    675 non-null    object 
 1   university_name                  675 non-null    object 
 2   year                             675 non-null    int64  
 3   world_rank                       675 non-null    float64
 4   national_rank                    675 non-null    float64
 5   overall_score                    339 non-null    float64
 6   country                          675 non-null    object 
 7   region                           675 non-null    object 
 8   city                             675 non-null    object 
 9   university_type                  675 non-null    object 
 10  Academic Reputation Score        675 non-null    float64
 11  Employer Reputation Score        675 non-null    float64
 12  citations_score       

,university_id,university_name,year,world_rank,national_rank,overall_score,country,region,city,university_type,...,country_avg_rank,universities_ranked_count,best_university_rank,country_avg_overall_score,country_avg_academic_reputation,country_avg_citations,country_avg_international_ratio,Teaching,Research Environment,Research Quality
0,U0001,University of Cambridge,2024,4.0,1.0,99.2,United Kingdom,Europe,Cambridge,Public,...,455.434783,69,1.0,50.817341,25.966667,388.050000,48.5,95.8,100.0,98.0
1,U0002,University of Oxford,2024,5.0,2.0,98.9,United Kingdom,Europe,Oxford,Public,...,455.434783,69,1.0,50.817341,25.966667,388.050000,48.5,96.6,100.0,99.0
2,U0003,Harvard University,2024,1.0,1.0,98.3,United States,North America,Cambridge,Private,...,362.265306,98,2.0,56.843245,31.065306,265.666667,23.5,97.7,99.9,99.4
3,U0004,Stanford University,2024,2.0,2.0,98.1,United States,North America,Stanford,Private,...,362.265306,98,2.0,56.843245,31.065306,265.666667,23.5,99.0,97.8,99.6
4,U0005,Imperial College London,2024,30.0,4.0,97.8,United Kingdom,Europe,London,Public,...,455.434783,69,1.0,50.817341,25.966667,388.050000,48.5,90.9,95.5,98.6


In [75]:
# Export cleanly structured result
df_output.to_csv("university_cleaned.csv", index=False)
print("Final 38-column file 'university_cleaned.csv' successfully saved and prepared for dashboard calculations!")

Final 38-column file 'university_cleaned.csv' successfully saved and prepared for dashboard calculations!
